# 00 Utils

Shared helpers: data loaders, firm name rules, product proximity, plot style.
Other notebooks load this one with `%run`, so it does not need to be run by hand.

Firm names: Manufacturer / applicant name harmonization across PQR and WHO PQ.

Two levels:
  firm  = one manufacturing company (legal-entity variants and spelling variants merged)
  group = corporate parent, for mergers/subsidiaries (e.g. Matrix -> Mylan, Lek -> Sandoz)
hq_country = country of the firm's headquarters, researcher-coded from public knowledge
             (needs spot-checking before it is used as an analytic variable).

Rules are applied in order; the first regex that matches the lower-cased raw name wins.
Names that match no rule fall back to a normalized key, so trivial variants
("Laboratorios Liconsa S.A." / "Laboratorios Liconsa SA") still merge.

In [ ]:
import itertools, re
from pathlib import Path
import pandas as pd, matplotlib
import matplotlib.pyplot as plt

## Data loaders

In [ ]:
# notebooks run from their own folder, so look upward for the project root
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
RAW = ROOT / "data" / "raw"
MED_CATS = {
    "Anti-Retroviral and COVID-19 medicine": "HIV/ARV",
    "Anti-TB medicine": "Tuberculosis",
    "Anti-malaria medicine": "Malaria",
}


def load_pqr(medicines_only=True):
    df = pd.read_csv(f"{RAW}/pqr_transactions_2026-10-10.csv", sep="\t",
                     encoding="utf-16", dtype=str)
    df.columns = [c.strip() for c in df.columns]
    df = df[df["Country/Teritorry"] != "Grand Total"].copy()
    df = df.rename(columns={
        "Country/Teritorry": "country", "Grant Name": "grant",
        "Supplier/Agent/Manufacturer/Intermediatry": "channel",
        "Manufacturer": "manufacturer", "Product Category": "category",
        "Product": "product", "Purchase Order Date": "po_date",
        "Pack quantity": "pack_qty", "Product pack (USD)": "pack_price_usd",
        "Total Product Cost (USD)": "cost_usd", "Primary Key": "pk",
    })
    for c in ["manufacturer", "product", "channel", "country"]:
        df[c] = df[c].str.strip()
    df["po_date"] = pd.to_datetime(df["po_date"], format="%d-%b-%y", errors="coerce")
    df["year"] = df["po_date"].dt.year
    for c in ["pack_qty", "pack_price_usd", "cost_usd"]:
        df[c] = pd.to_numeric(df[c].str.replace(",", ""), errors="coerce")
    if medicines_only:
        df = df[df["category"].isin(MED_CATS)].copy()
        df["area"] = df["category"].map(MED_CATS)
    return df


def _pq_date(s):
    s = s.astype(str).str.replace(r"\s+", " ", regex=True).str.replace(",", "")
    return pd.to_datetime(s, format="%d %b %Y", errors="coerce")


def load_pq_fpp():
    d = pd.read_csv(f"{RAW}/who_pq_fpp_2026-10-10.csv")
    d["pq_date"] = _pq_date(d["Date of Prequalification"])
    d["year"] = d["pq_date"].dt.year
    return d


def load_pq_api():
    d = pd.read_csv(f"{RAW}/who_pq_api_2026-10-10.csv")
    d["pq_date"] = _pq_date(d["Date of prequalification"])
    d["year"] = d["pq_date"].dt.year
    return d

## Firm name rules

In [ ]:
# (regex on lowercased raw name, firm, group, hq_country)
RULES = [
    # --- multinational originators
    (r"abbvie|abbott|^abbsp", "AbbVie", "AbbVie", "USA"),
    (r"astra ?zeneca", "AstraZeneca", "AstraZeneca", "UK"),
    (r"bayer", "Bayer", "Bayer", "Germany"),
    (r"boehringer|^bi, ingelheim",  "Boehringer Ingelheim", "Boehringer Ingelheim", "Germany"),
    (r"bristol|^bms\b",  "Bristol-Myers Squibb", "Bristol-Myers Squibb", "USA"),
    (r"lilly", "Eli Lilly", "Eli Lilly", "USA"),
    (r"gilead", "Gilead Sciences", "Gilead Sciences", "USA"),
    (r"janssen|tibotec", "Janssen", "Johnson & Johnson", "Belgium"),
    (r"merck sharp|\bmsd\b", "Merck Sharp & Dohme", "Merck & Co", "USA"),
    (r"merck (europe|healthcare kgaa)", "Merck KGaA", "Merck KGaA", "Germany"),
    (r"novartis", "Novartis", "Novartis", "Switzerland"),
    (r"novo nordisk", "Novo Nordisk", "Novo Nordisk", "Denmark"),
    (r"otsuka", "Otsuka", "Otsuka", "Japan"),
    (r"pfizer", "Pfizer", "Pfizer", "USA"),
    (r"\broche\b|hoffman", "Roche", "Roche", "Switzerland"),
    (r"sanofi", "Sanofi", "Sanofi", "France"),
    (r"viiv", "ViiV Healthcare", "GSK", "UK"),
    (r"^gsk\b", "ViiV Healthcare", "GSK", "UK"),   # SCMS manufacturing sites of GSK produce ViiV ARVs
    (r"glaxo", "GlaxoSmithKline", "GSK", "UK"),
    (r"organon", "Organon", "Organon", "USA"),
    (r"shionogi", "Shionogi", "Shionogi", "Japan"),
    (r"eisai", "Eisai", "Eisai", "Japan"),
    (r"regeneron", "Regeneron", "Regeneron", "USA"),
    (r"chiesi", "Chiesi", "Chiesi", "Italy"),
    (r"ferring", "Ferring", "Ferring", "Switzerland"),
    (r"mcneil", "McNeil", "Johnson & Johnson", "Sweden"),
    # --- generic multinationals and their subsidiaries
    (r"sandoz", "Sandoz", "Sandoz", "Switzerland"),
    (r"\blek pharm", "Lek Pharmaceuticals", "Sandoz", "Slovenia"),
    (r"\bteva\b", "Teva", "Teva", "Israel"),
    (r"actavis", "Actavis", "Teva", "Ireland"),
    (r"barr lab", "Barr Laboratories", "Teva", "USA"),
    (r"\bmepha\b", "Mepha", "Teva", "Switzerland"),
    (r"mylan|matrix laboratories", "Mylan", "Viatris", "India"),
    (r"pharmacare|aspen", "Aspen Pharmacare", "Aspen", "South Africa"),
    (r"apotex", "Apotex", "Apotex", "Canada"),
    (r"medreich", "Medreich", "Meiji", "India"),
    (r"^meiji", "Meiji", "Meiji", "Japan"),
    (r"riemser|fatol", "Riemser", "Riemser", "Germany"),
    (r"dafra", "Dafra Pharma", "Dafra Pharma", "Belgium"),
    (r"sigma tau|alfasigma", "Alfasigma", "Alfasigma", "Italy"),
    (r"reig jof", "Reig Jofre", "Reig Jofre", "Spain"),
    (r"liconsa", "Liconsa", "Liconsa", "Spain"),
    (r"labesfal", "Labesfal", "Fresenius Kabi", "Portugal"),
    (r"antibiotice", "Antibiotice", "Antibiotice", "Romania"),
    (r"^demo s", "Demo SA", "Demo SA", "Greece"),
    (r"celltrion", "Celltrion", "Celltrion", "South Korea"),
    (r"dong-?a", "Dong-A", "Dong-A", "South Korea"),
    (r"shin poong", "Shin Poong", "Shin Poong", "South Korea"),
    # --- India
    (r"quality chemical", "Quality Chemical Industries", "Quality Chemical Industries", "Uganda"),
    (r"aurobindo", "Aurobindo", "Aurobindo", "India"),
    (r"cipla", "Cipla", "Cipla", "India"),
    (r"hetero", "Hetero", "Hetero", "India"),
    (r"macleods", "Macleods", "Macleods", "India"),
    (r"lupin", "Lupin", "Lupin", "India"),
    (r"laurus", "Laurus Labs", "Laurus Labs", "India"),
    (r"micro labs", "Micro Labs", "Micro Labs", "India"),
    (r"\bipca\b", "Ipca", "Ipca", "India"),
    (r"strides", "Strides", "Strides", "India"),
    (r"^sun pharma|ranbaxy", "Sun Pharma", "Sun Pharma", "India"),
    (r"\bmsn\b", "MSN Laboratories", "MSN", "India"),
    (r"emcure", "Emcure", "Emcure", "India"),
    (r"ajanta", "Ajanta Pharma", "Ajanta Pharma", "India"),
    (r"cadila pharmaceuticals", "Cadila Pharmaceuticals", "Cadila Pharmaceuticals", "India"),
    (r"zydus", "Zydus Lifesciences", "Zydus", "India"),
    (r"intas", "Intas", "Intas", "India"),
    (r"svizera", "Svizera", "Svizera", "India"),
    (r"s kant", "S Kant Healthcare", "S Kant Healthcare", "India"),
    (r"medopharm", "Medopharm", "Medopharm", "India"),
    (r"optimus drugs", "Optimus Drugs", "Optimus", "India"),
    (r"styrax", "Styrax", "Styrax", "India"),
    (r"piramal", "Piramal", "Piramal", "India"),
    (r"dr reddy", "Dr Reddy's", "Dr Reddy's", "India"),
    (r"gland pharma", "Gland Pharma", "Fosun", "India"),
    (r"serum institute", "Serum Institute of India", "Serum Institute of India", "India"),
    (r"remington", "Remington", "Remington", "Pakistan"),
    (r"universal corporation", "Universal Corporation", "Universal Corporation", "Kenya"),
    # --- China
    (r"guilin", "Guilin Pharmaceutical", "Fosun", "China"),
    (r"shanghai desano", "Shanghai Desano", "Desano", "China"),
    (r"jiangxi desano", "Jiangxi Desano", "Desano", "China"),
    (r"^desano pharmaceuticals private", "Desano India", "Desano", "India"),
    (r"zizhu", "China Resources Zizhu", "China Resources", "China"),
    (r"san jiu", "China Resources Sanjiu", "China Resources", "China"),
    (r"holley", "Holley-Cotec", "Holley-Cotec", "China"),
    (r"huahai", "Zhejiang Huahai", "Zhejiang Huahai", "China"),
    (r"reyoung", "Shandong Reyoung", "Shandong Reyoung", "China"),
    (r"hisun", "Zhejiang Hisun", "Zhejiang Hisun", "China"),
    (r"north china pharm|ncpc", "North China Pharmaceutical", "North China Pharmaceutical", "China"),
    (r"shijiazhuang pharma group", "CSPC", "CSPC", "China"),
    (r"lonzeal", "Shijiazhuang Lonzeal", "Shijiazhuang Lonzeal", "China"),
    (r"shanghai harvest", "Shanghai Harvest", "Shanghai Harvest", "China"),
    (r"anhui biochem", "Anhui Biochem", "Anhui Biochem", "China"),
]

# Country heuristic for names with no rule: Chinese place names, Indian legal forms.
CN_PLACES = r"\b(anhui|beijing|changzhou|chongqing|guangdong|hainan|hebei|hunan|jiangsu|jiangxi|liaoyuan|qidong|qilu|qinhuangdao|shandong|shanghai|shaoxing|shenzhen|shijiazhuang|shouguang|zhejiang|zhuhai)\b"
IN_FORMS = r"\b(private limited|pvt\.? ?ltd|\(p\) ltd|pvt\b)|\b(india|mumbai)\b"


def country_heuristic(raw):
    low = raw.lower()
    if re.search(CN_PLACES, low):
        return "China"
    if re.search(IN_FORMS, low):
        return "India"
    return None

LEGAL = r"\b(ltd|limited|inc|pvt|private|co|company|corp|corporation|llc|plc|sa|s a|spa|s p a|gmbh|kg|kgaa|ag|bv|b v|nv|pty|pte|uc|ab|as|oy|jsc|d d|srl|fzc|the|m s)\b"


def norm_key(raw):
    s = raw.lower()
    s = re.sub(r"\(.*?\)", " ", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\blaboratories\b|\blaboratory\b", "labs", s)
    s = re.sub(r"\bpharmaceuticals?\b", "pharma", s)
    s = re.sub(LEGAL, " ", s)
    return re.sub(r"\s+", " ", s).strip()


def match(raw):
    """Return (firm, group, hq_country, method) for one raw name, or None for junk."""
    if raw is None or raw.startswith("_Info"):
        return None
    low = raw.lower()
    for pat, firm, group, cty in RULES:
        if re.search(pat, low):
            return firm, group, cty, "rule"
    return None, None, None, "normalized"

## Product proximity (Hidalgo et al. 2007)

In [ ]:
def project_products(e):
    """Return product proximity edge list (phi) from a firm-product edge list."""
    k = e.groupby("product")["firm"].nunique()
    pairs = {}
    for _, prods in e.groupby("firm")["product"]:
        for a, b in itertools.combinations(sorted(set(prods)), 2):
            pairs[(a, b)] = pairs.get((a, b), 0) + 1
    rows = [(a, b, c, c / max(k[a], k[b])) for (a, b), c in pairs.items()]
    return pd.DataFrame(rows, columns=["source", "target", "shared_firms", "phi"])

## Plot style

In [ ]:
def set_style(font_size=10, grid=True):
    """Shared matplotlib style for all figures."""
    GRID = "#e6e5e1"
    plt.rcParams.update({"font.family": ["Trebuchet MS", "DejaVu Sans"], "font.size": font_size, "axes.edgecolor": GRID,
        "axes.labelcolor": "#595959", "xtick.color": "#595959", "ytick.color": "#595959", "text.color": "#0b0b0b",
        "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 200, "savefig.bbox": "tight"})
    if grid:
        plt.rcParams.update({"axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True})